In [27]:
'''
abbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),
    SMBHB (supermassive black hole binary)
'''

'\nabbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),\n    SMBHB (supermassive black hole binary)\n'

In [28]:
import os
import otter

import astropy
from astropy.coordinates import SkyCoord
from astropy import units as u

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dustmaps.config import config
config.reset()

from sparcl.client import SparclClient


In [29]:
db = otter.Otter()

Attempting to login to https://otter.idies.jhu.edu/api with the following credentials:
username: user-guest
password: test


In [30]:
# candidate list quick reference table
cands = db.query(classification="TDE", minz=0, maxz=0.03)

#initialize all lists; each one to be made into an array to be stored in dataframe
name_listi = []
hosts_booleani = []
hostnamelist = []
confidencei, referencesi = [], []

# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_listi.append(name)
    
    info = object.get_host()
    if info == []:
        hosts_booleani.append('False')
        hostnamelist.append('N/A')
    else:
        info = info[0]
        hosts_booleani.append('True')
        hostnamelist.append(info.name)

    classification, conf, refs = object.get_classification()
    confidencei.append(conf)
    bibcodes = []
    for b in refs:
        bibcodes.append(b)
    referencesi.append(bibcodes)
        
    i += 1

    
# initialize data frame storing all possible canidates
quickref_df = pd.DataFrame(
    {"TDE Name": name_listi, 'References (Bibcodes)': referencesi, 'Confidence': confidencei, "Host info in OTTER?": hosts_booleani, "Host Name": hostnamelist}
)



quickref_df

,TDE Name,References (Bibcodes),Confidence,Host info in OTTER?,Host Name
0,2MASX J01190869-3411305,"[2023PASP..135c4101G, 2019Natur.573..381M]",1.0,True,GSN 069
1,UGC 03317,"[2016A&A...586A...9H, 2017ApJ...835...64G, 202...",1.0,True,UGC 3317
2,PGC 170392,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",1.0,True,6dFGS gJ222646.4-150123
3,PGC 1127938,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",1.0,True,2SLAQ J011844.35-010906.8
4,PGC 015259,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",1.0,True,2MFGC 3645
5,NGC 5905,"[1996A&A...309L..35B, 2017ApJ...838..149A, 202...",3.1,True,NGC 5905
6,NGC 247,"[2011ApJ...737...87J, 2015ApJ...807..185F, 201...",1.0,True,NGC 247
7,PGC 1185375,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",1.0,True,2MASX J15035028+0107366
8,IGR J17361-4441,"[2011A&A...535L...1B, 2014arXiv1407.5081D, 201...",1.0,True,NGC 6388
9,XMMSL1 J131952.3+225958,"[2023PASP..135c4101G, 2020ApJ...891..121L]",1.0,True,NGC 5092


In [31]:
# MASTER TABLE
# initial query - creates canidate list as objects classified as TDEs and within redshift 0.03
cands = db.query(classification="TDE", minz=0, maxz=0.03)

#initialize all lists; each one to be made into an array to be stored in dataframe
name_list, lum_list, rs_list = [], [], []
cord_ra_list, cord_dec_list = [], []
confidence, references = [], []
hosts_boolean, host_names, hosts_cords = [], [], []


# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_list.append(name)

    
    dists = pd.DataFrame(object['distance']) # find all possible distance measurments
    # luminosity
    lum_dists = dists[dists.distance_type == 'luminosity']
    if len(lum_dists) != 0:
        obj_lums = lum_dists['value'].tolist()
    else:
        obj_lums = 'Empty'
    lum_list.append(obj_lums)
    # redshift
    rs_dists = dists[dists.distance_type == 'redshift']
    obj_rs = (rs_dists['value']).tolist()
    rs_list.append(obj_rs)

    # ra and dec of transient in deg
    cords = object.get_skycoord()
    ra = cords.ra.deg
    cord_ra_list.append(ra)
    dec = cords.dec.deg
    cord_dec_list.append(dec)
    
    # confidence transient is a TDE
    classification, conf, refs = object.get_classification()
    confidence.append(conf)
    bibcodes = []
    for b in refs:
        bibcodes.append(b)
    references.append(bibcodes)
    
    info = object.get_host()
    if info == []:
        hosts_boolean.append('False')
        host_names.append('N/A')
        hosts_cords.append('N/A')
        ztf_lens.append('N/A')
    else:
        info = info[0]
        hosts_boolean.append('True')
        host_names.append(info.name)
        hosts_cords.append(info.coord)

        
    i += 1

    
# initialize data frame storing all possible canidates
cands_df = pd.DataFrame(
    {"TDE Name": name_list, 'References (Bibcodes)': references, "default redshift": [item[0] for item in rs_list], "redshifts": rs_list, "luminosity (Mpc)": lum_list, 
     "confidence": confidence, "ra (deg)": cord_ra_list, "dec (deg)":cord_dec_list, "Host info in OTTER?": hosts_boolean, 
     "Host Name": host_names, "Host Coords (deg)": hosts_cords}
)




cands_df

,TDE Name,References (Bibcodes),default redshift,redshifts,luminosity (Mpc),confidence,ra (deg),dec (deg),Host info in OTTER?,Host Name,Host Coords (deg)
0,2MASX J01190869-3411305,"[2023PASP..135c4101G, 2019Natur.573..381M]",0.018,[0.018],Empty,1.0,19.783333,-34.191667,True,GSN 069,"<SkyCoord (ICRS): (ra, dec) in deg\n (19.78..."
1,UGC 03317,"[2016A&A...586A...9H, 2017ApJ...835...64G, 202...",0.004136,[0.004136],[18.36],1.0,83.407000,73.724000,True,UGC 3317,"<SkyCoord (ICRS): (ra, dec) in deg\n (83.40..."
2,PGC 170392,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",0.016246,[0.016246],[72.791],1.0,336.693000,-15.023000,True,6dFGS gJ222646.4-150123,"<SkyCoord (ICRS): (ra, dec) in deg\n (336.6..."
3,PGC 1127938,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",0.02,[0.02],[90],1.0,19.736000,-1.053000,True,2SLAQ J011844.35-010906.8,"<SkyCoord (ICRS): (ra, dec) in deg\n (19.68..."
4,PGC 015259,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",0.014665,[0.014665],[65.629],1.0,67.341000,-4.760000,True,2MFGC 3645,"<SkyCoord (ICRS): (ra, dec) in deg\n (67.34..."
5,NGC 5905,"[1996A&A...309L..35B, 2017ApJ...838..149A, 202...",0.01124,"[0.01124, 0.012]",[50.17],3.1,228.847167,55.517108,True,NGC 5905,"<SkyCoord (ICRS): (ra, dec) in deg\n (228.8..."
6,NGC 247,"[2011ApJ...737...87J, 2015ApJ...807..185F, 201...",0.000531,[0.000531],[2.35],1.0,11.786000,-20.760000,True,NGC 247,"<SkyCoord (ICRS): (ra, dec) in deg\n (11.78..."
7,PGC 1185375,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",0.00523,[0.00523],[23.2],1.0,225.960000,1.127000,True,2MASX J15035028+0107366,"<SkyCoord (ICRS): (ra, dec) in deg\n (225.9..."
8,IGR J17361-4441,"[2011A&A...535L...1B, 2014arXiv1407.5081D, 201...",0.000271,[0.000271],[1.2],1.0,264.072583,-44.734994,True,NGC 6388,"<SkyCoord (ICRS): (ra, dec) in deg\n (264.0..."
9,XMMSL1 J131952.3+225958,"[2023PASP..135c4101G, 2020ApJ...891..121L]",0.023,[0.023],Empty,1.0,199.966667,22.999444,True,NGC 5092,"<SkyCoord (ICRS): (ra, dec) in deg\n (199.9..."


In [32]:
print(cands_df.columns.tolist())

['TDE Name', 'References (Bibcodes)', 'default redshift', 'redshifts', 'luminosity (Mpc)', 'confidence', 'ra (deg)', 'dec (deg)', 'Host info in OTTER?', 'Host Name', 'Host Coords (deg)']


In [34]:
hostless_cands_df = cands_df.loc[cands_df["Host info in OTTER?"] == 'False', ["TDE Name", 'References (Bibcodes)', "default redshift", 
                                                                            "redshifts", "luminosity (Mpc)", "confidence", "ra (deg)", 
                                                                           "dec (deg)", "Host info in OTTER?"]].copy()

hostless_cands_df

,TDE Name,References (Bibcodes),default redshift,redshifts,luminosity (Mpc),confidence,ra (deg),dec (deg),Host info in OTTER?
11,VT J1356,[2025ApJ...982..163S],0.018,[0.018],Empty,1.0,209.050417,-26.980750,False
15,ARP 299-B AT1,[2018Sci...361..482M],0.010411,[0.010411],Empty,3.3,172.129115,58.561329,False
19,IGR J12580+0134,"[2015ApJ...809..172I, 2016MNRAS.461.3375Y, 201...",0.00411,[0.00411],Empty,1.0,194.521250,1.573889,False
22,WTP14adbwvs,"[2020ApJ...896L..27D, 2024ApJ...961..211M]",0.02489,[0.02489],[111.1],1.0,198.748944,51.272774,False
24,2022gri,"[2022TNSAN..99....1Y, 2022TNSCR1156....1Y, 202...",0.028,[0.028],Empty,3.3,109.586643,33.994853,False
25,2022dsb,"[2022TNSCR.584....1F, 2024ApJ...974..241A, 202...",0.023,"[0.023, 0.0234]",Empty,3.3,235.590633,-22.670580,False
27,2020wey,"[2020TNSCR3228....1A, 2023ApJ...942....9H, 202...",0.02738,"[0.02738, 0.02741]",Empty,3.3,136.357958,61.802538,False
28,AT2019ahk,"[2019ApJ...883..111H, 2024ApJ...974..241A, 202...",0.026,"[0.026, 0.0262]",Empty,3.3,105.047461,-66.040214,False
29,AT2022mim,[2024ApJ...961..211M],0.02035,[0.02035],[89.3],3.2,251.477888,-23.451942,False
30,2022dbl,"[2022TNSCR.504....1A, 2024MNRAS.527.2452M]",0.0284,[0.0284],Empty,3.3,185.187825,49.551222,False


In [45]:
host_cands_df = cands_df.loc[cands_df["Host info in OTTER?"] == 'True', ["TDE Name", 'References (Bibcodes)', 'Host Name', "confidence", "ra (deg)", 
                                                                           "dec (deg)", 'Host Coords (deg)']].copy()

def query_ztf_helper(transient):
    # where argument 'transient' is a string of the default name of the transient object
    obj = db.query(names=transient)[0]
    host_obj = obj.get_host()[0]

    ztf_info = host_obj.query_ztf()
    return ztf_info


host_cands_df['ZTF table len'] = host_cands_df['TDE Name'].apply(lambda x: len(query_ztf_helper(x)))



host_cands_df

,TDE Name,References (Bibcodes),Host Name,confidence,ra (deg),dec (deg),Host Coords (deg),ZTF table len
0,2MASX J01190869-3411305,"[2023PASP..135c4101G, 2019Natur.573..381M]",GSN 069,1.0,19.783333,-34.191667,"<SkyCoord (ICRS): (ra, dec) in deg\n (19.78...",0
1,UGC 03317,"[2016A&A...586A...9H, 2017ApJ...835...64G, 202...",UGC 3317,1.0,83.407000,73.724000,"<SkyCoord (ICRS): (ra, dec) in deg\n (83.40...",81
2,PGC 170392,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",6dFGS gJ222646.4-150123,1.0,336.693000,-15.023000,"<SkyCoord (ICRS): (ra, dec) in deg\n (336.6...",11
3,PGC 1127938,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",2SLAQ J011844.35-010906.8,1.0,19.736000,-1.053000,"<SkyCoord (ICRS): (ra, dec) in deg\n (19.68...",4
4,PGC 015259,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",2MFGC 3645,1.0,67.341000,-4.760000,"<SkyCoord (ICRS): (ra, dec) in deg\n (67.34...",1907
5,NGC 5905,"[1996A&A...309L..35B, 2017ApJ...838..149A, 202...",NGC 5905,3.1,228.847167,55.517108,"<SkyCoord (ICRS): (ra, dec) in deg\n (228.8...",3571
6,NGC 247,"[2011ApJ...737...87J, 2015ApJ...807..185F, 201...",NGC 247,1.0,11.786000,-20.760000,"<SkyCoord (ICRS): (ra, dec) in deg\n (11.78...",11
7,PGC 1185375,"[2016A&A...586A...9H, 2017ApJ...835...64G, 201...",2MASX J15035028+0107366,1.0,225.960000,1.127000,"<SkyCoord (ICRS): (ra, dec) in deg\n (225.9...",1665
8,IGR J17361-4441,"[2011A&A...535L...1B, 2014arXiv1407.5081D, 201...",NGC 6388,1.0,264.072583,-44.734994,"<SkyCoord (ICRS): (ra, dec) in deg\n (264.0...",0
9,XMMSL1 J131952.3+225958,"[2023PASP..135c4101G, 2020ApJ...891..121L]",NGC 5092,1.0,199.966667,22.999444,"<SkyCoord (ICRS): (ra, dec) in deg\n (199.9...",6


In [50]:
hosts_mini_df = host_cands_df[["TDE Name", 'Host Name']].copy()

hosts_mini_df

,TDE Name,Host Name
0,2MASX J01190869-3411305,GSN 069
1,UGC 03317,UGC 3317
2,PGC 170392,6dFGS gJ222646.4-150123
3,PGC 1127938,2SLAQ J011844.35-010906.8
4,PGC 015259,2MFGC 3645
5,NGC 5905,NGC 5905
6,NGC 247,NGC 247
7,PGC 1185375,2MASX J15035028+0107366
8,IGR J17361-4441,NGC 6388
9,XMMSL1 J131952.3+225958,NGC 5092
